Task 2:

Loading in all the datasets.

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

#Load the Training Dataset
df = pd.read_csv("community_risk_screening_train.csv")

#Select the Model Features
feature_columns = [
    "age_at_screening",
    "region",
    "housing_status",
    "prior_crisis_events_12mo",
    "phq2_score",
    "substance_use_flag",
    "primary_support_type",
    "days_since_last_contact"
]

#Separate Features and Target
x = df[feature_columns]
y = df["crisis_within_90_days"]

#Split into Training (70%) and Temp (30%)  Data
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.30, random_state=42, stratify=y)

#Split Temp Data into Validation (15%) and Test (15%)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

#Select Numeric and Categorical Features
numeric_features = [
    "age_at_screening",
    "prior_crisis_events_12mo",
    "phq2_score",
    "days_since_last_contact"
]

categorical_features = [
    "region",
    "housing_status",
    "substance_use_flag",
    "primary_support_type"
]

#Preprocessing for Numeric Features
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])
#Preprocessing for Categorical Features
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

#Combine the Preprocessing Steps
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

#Create the Selected Random Forest Model
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight="balanced",
        random_state=42
    ))
])

#Train the Model
rf_pipeline.fit(x_train, y_train)

#Get the Probability of Crisis Prediction
crisis_index = list(rf_pipeline.classes_).index("crisis")
val_probabilities = rf_pipeline.predict_proba(x_val)[:, crisis_index]

#Apply the Threshold Picked from Week 4
val_predictions = (val_probabilities >= 0.30).astype(int)

#Convert Actual Validation Outcomes to Binary Values
y_val_binary = (y_val == "crisis").astype(int)

#Create a Copy of Validation Features
validation_audit = x_val.copy()

#Add Actual and Predicted Results
validation_audit["actual_crisis"] = y_val_binary
validation_audit["predicted_crisis"] = pd.Series(
    val_predictions, index=x_val.index
)

#Create Age Groups for Fairness Audit
validation_audit["age_group"] = pd.cut(
    validation_audit["age_at_screening"],
    bins=[17, 29, 44, 59, 74, 89],
    labels=[
        "Young Adult",
        "Adult",
        "Middle-Aged Adult",
        "Older Adult",
        "Senior Adult"
    ]
).astype("object")

#Keep Missing Age Values as Their Own Group
validation_audit["age_group"] = validation_audit["age_group"].fillna(
    "Age Group Unknown"
)

#Label Missing Values in Other Subgroup Features
subgroup_features = [
    "region",
    "housing_status",
    "substance_use_flag"
]

for feature in subgroup_features:
    validation_audit[feature] = (
        validation_audit[feature].fillna("Unknown")
    )

#Display Dataset
print("Training Set:", len(x_train))
print("Validation Set:", len(x_val))
print("Test Set:", len(x_test))
print("Random Forest Model Trained Successfully!")
print("Model Classes:", rf_pipeline.classes_)
print("Validation Predictions:", len(val_predictions))
print("Actual Crisis Cases:", y_val_binary.sum())
print("Predicted Crisis Cases:", val_predictions.sum())
for feature in ["region", "age_group", "housing_status", "substance_use_flag"]:
    print(f"\n{feature} Subgroups:")

    subgroup_counts = validation_audit.groupby(
        feature, observed=True
    )["actual_crisis"].agg(
        Total_Clients="count",
        Crisis_Cases="sum"
    )

    print(subgroup_counts)

Training Set: 1680
Validation Set: 360
Test Set: 360
Random Forest Model Trained Successfully!
Model Classes: ['crisis' 'no_crisis']
Validation Predictions: 360
Actual Crisis Cases: 53
Predicted Crisis Cases: 79

region Subgroups:
               Total_Clients  Crisis_Cases
region                                    
Delta Basin               17             4
Gulf Coast               120            19
Inland Valley            116            15
Metro North              107            15

age_group Subgroups:
                   Total_Clients  Crisis_Cases
age_group                                     
Adult                         59             1
Age Group Unknown             27             2
Middle-Aged Adult             84            12
Older Adult                   60             8
Senior Adult                  96            28
Young Adult                   34             2

housing_status Subgroups:
                      Total_Clients  Crisis_Cases
housing_status                      

Initial Fairness Audit

In [9]:
from sklearn.metrics import recall_score, precision_score, f1_score

#Store the Subgroup Results
subgroup_results = []

#Analyze the Approved Subgroup
for feature in ["region", "age_group", "housing_status", "substance_use_flag"]:

    for group_name, group_data in validation_audit.groupby(feature, observed=True):

        #Get Actual and Predicted Outcomes
        actual = group_data["actual_crisis"]
        predicted = group_data["predicted_crisis"]

        #Count Clients and Actual Crisis Cases
        total_clients = len(group_data)
        crisis_case = int(actual.sum())

        #Calculate Performance Metrics
        recall = recall_score(actual, predicted, zero_division=0)
        precision = precision_score(actual, predicted, zero_division=0)
        f1 = f1_score(actual, predicted, zero_division=0)

        #Calculate False Negative Rate
        false_negative_rate = 1 - recall

        #Add a Reliability Note for Small Groups
        if crisis_case < 20:
            reliability_note = "Low Sample, Interpret with Caution"
        else:
            reliability_note = "Sufficient Sample for this Audit"

        #Save Results
        subgroup_results.append({
            "Subgroup Feature": feature,
            "Group": group_name,
            "Total Clients": total_clients,
            "Crisis Cases": crisis_case,
            "Recall": recall,
            "False Negative Rate": false_negative_rate,
            "Precision": precision,
            "F1 Score": f1,
            "Reliability Note": reliability_note
    })

#Create Results Table
subgroup_metrics = pd.DataFrame(subgroup_results)

#Display Results
print("Week 5 Subgroup Fairness Audit Results:")
display(subgroup_metrics.round(3))

Week 5 Subgroup Fairness Audit Results:


,Subgroup Feature,Group,Total Clients,Crisis Cases,Recall,False Negative Rate,Precision,F1 Score,Reliability Note
0,region,Delta Basin,17,4,0.750,0.250,0.600,0.667,"Low Sample, Interpret with Caution"
1,region,Gulf Coast,120,19,0.684,0.316,0.619,0.650,"Low Sample, Interpret with Caution"
2,region,Inland Valley,116,15,0.933,0.067,0.467,0.622,"Low Sample, Interpret with Caution"
3,region,Metro North,107,15,0.800,0.200,0.522,0.632,"Low Sample, Interpret with Caution"
4,age_group,Adult,59,1,0.000,1.000,0.000,0.000,"Low Sample, Interpret with Caution"
5,age_group,Age Group Unknown,27,2,1.000,0.000,0.286,0.444,"Low Sample, Interpret with Caution"
6,age_group,Middle-Aged Adult,84,12,0.667,0.333,0.571,0.615,"Low Sample, Interpret with Caution"
7,age_group,Older Adult,60,8,0.750,0.250,0.545,0.632,"Low Sample, Interpret with Caution"
8,age_group,Senior Adult,96,28,0.929,0.071,0.619,0.743,Sufficient Sample for this Audit
9,age_group,Young Adult,34,2,0.000,1.000,0.000,0.000,"Low Sample, Interpret with Caution"


Initial Fairness Audit Results:

### Recall Gap Results

| Subgroup Category | Highest Recall Group | Lowest Recall Group | Recall Gap | Audit Result |
|---|---|---|---:|---|
| Region | Inland Valley (0.933) | Gulf Coast (0.684) | 24.9 percentage points | Flagged |
| Age Group | Senior Adult (0.929) | Adult / Young Adult (0.000) | 92.9 percentage points | Flagged |
| Housing Status | Unknown / Doubled Up (1.000) | Stable Housed (0.438) | 56.2 percentage points | Flagged |
| Substance Use Flag | False (0.840) | True (0.750) | 9.0 percentage points | Not Flagged |
Note:
- Recall Gap Threshold: Difference greater than 10% points are flagged for further investigation.
- Several subgroups have fewer than 20 actual crisis cases. These recall results should be interpreted with caution.

Subgroup Results:
- Region: Inland Valley has the highest recall at 93.3%, while Gulf Coast has the lowest at 68.4%. They have a total of 24.9% point gap.
- Age Group: Senior Adults have a recall of 92.9%, while Adults and Young Adults have a recall of 0%. The Adults and Young Adults only have 1 and 2 actual crisis cases, so these results are uncertain.
- Housing Status: Doubled up clients have a recall of 100%, while stable housed clients have a recall of 43.8%. This is a substantial difference but both groups have fewer than 20 actual crisis cases.
- Substance Use Flag: The recall difference between clients with and without a recorded substance use concern is 9% points, which is below our 10 point threshold.



Fairness Audit Summary and Interpretation:
The initial fairness audit identified the recall differences across several subgroups. Region, age group, and housing status exceeded the approved 10% point recall gap threshold. However, substance use flag didn't exceed the threshold with a 9% point difference.

The largest difference is in age group and housing status, but alot of the subgroups have fewer than 20 actual crisis cases, so these results should be interpreted with caution. These findings do not automatically indicate that the model is unfair. Instead, they show us areas that need further investigation before any consideration.

Age Group False Negative Analysis

In [11]:
#Examine False Negatives by Age Group
age_group_errors = validation_audit.groupby(
    "age_group", observed=True).apply(
    lambda group: pd.Series({
        "Total Crisis Cases": int(group["actual_crisis"].sum()),
        "Correctly Identified": int(
            ((group["actual_crisis"] == 1) & (group["predicted_crisis"] == 1)).sum()
        ),
        "Missed Crisis Cases": int(
            ((group["actual_crisis"] == 1) & (group["predicted_crisis"] == 0)).sum()
        )
    })
)

print("Age Group False Negative Analysis:")
display(age_group_errors)

Age Group False Negative Analysis:


,Total Crisis Cases,Correctly Identified,Missed Crisis Cases
age_group,,,
Adult,1,0,1
Age Group Unknown,2,2,0
Middle-Aged Adult,12,8,4
Older Adult,8,6,2
Senior Adult,28,26,2
Young Adult,2,0,2


Age Group False Negative Analysis Results:


- Adults had only 1 actual crisis case, and the model missed it.
- Young Adults had only 2 actual crisis cases, and the model missed both.
- Senior Adults had 28 actual crisis cases, and the model correctly identified 26.
- Middle-Aged Adults had 12 crisis cases, with 4 missed.

The false negative analysis showed that the Adult and Young Adult groups had very small sample sizes in general. Adults only had 1 that is actual crisis case, and the Young Adults had 2 actual crisis cases. The model have missed all 3. Senior Adults have 28 actual crisis cases, and 2 were missed by the model. Middle Aged Adults had 12 actual crisis cases, and the model missed 4.

These results tell us that the small sample sizes in the Adult and Young Adult groups contribute to the large recall gap.

Age Group Fairness Audit Conclusion:

The age group recall gap of 92.9 exceed points is largely driven by the small sample sizes in the Adult and Young Adult groups. Since Adults only had 1 and Young Adults only had 2, the model missed all 3 cases which gave us a results of a recall of 0% for both groups. Since these groups have very few actual crisis cases, their recall results are not reliable enough for us to confidently determine whether the model is biased against them.

Region False Negative Analysis

In [16]:
#Examine False Negatives by Region
region_errors = validation_audit.groupby(
    "region", observed=True
).apply(
    lambda group: pd.Series({
        "Total Crisis Cases": int(group["actual_crisis"].sum()),

        "Correctly Identified": int(
            ((group["actual_crisis"] == 1) &
             (group["predicted_crisis"] == 1)).sum()
        ),

        "Missed Crisis Cases": int(
            ((group["actual_crisis"] == 1) &
             (group["predicted_crisis"] == 0)).sum()
        )
    })
)

print("Region False Negative Analysis Results:")
display(region_errors)

Region False Negative Analysis Results:


,Total Crisis Cases,Correctly Identified,Missed Crisis Cases
region,,,
Delta Basin,4,3,1
Gulf Coast,19,13,6
Inland Valley,15,14,1
Metro North,15,12,3


Region False Negative Analysis Results:


- Gulf Coast: Model missed 6 out of 19 actual crisis cases, largest number of missed cases.
- Inland Valley: Model missed 1 out of 15 actual crisis cases.
- Metro North: Model missed 3 out of 15 actual crisis cases.
- Delta Basin: Model missed 1 out of 4 actual crisis cases, however since this has a small sample size makes the results less reliable.

The region analysis shows us that Gulf Coast had the highest missed actual crisis case by the model. The difference between Inland Valley's recall of 93.3 percentage and Gulf Coast's recall of 68.4 percentage is a 24.9 percentage points difference, this exceed the 10 percentage point threshold, so this gap should be investigated further. All four regions also had fewer than 20 actual crisis cases in the validation set, so these results need to be interpreted with caution.

Housing Status False Negative Analysis

In [17]:
#Examine False Negatives by Housing Status
housing_errors = validation_audit.groupby(
    "housing_status", observed=True
).apply(
    lambda group: pd.Series({
        "Total Crisis Cases": int(group["actual_crisis"].sum()),

        "Correctly Identified": int(
            ((group["actual_crisis"] == 1) &
             (group["predicted_crisis"] == 1)).sum()
        ),

        "Missed Crisis Cases": int(
            ((group["actual_crisis"] == 1) &
             (group["predicted_crisis"] == 0)).sum()
        )
    })
)

print("Housing Status False Negative Analysis Results:")
display(housing_errors)

Housing Status False Negative Analysis Results:


,Total Crisis Cases,Correctly Identified,Missed Crisis Cases
housing_status,,,
Unknown,4,4,0
doubled_up,13,13,0
stable_housed,16,7,9
transitional,13,12,1
unstable_or_unhoused,7,6,1


Housing Status False Negative Analysis Results:


- Stable Housed: Model missed 9 out of 16 actual crisis cases.
- Doubled Up: Model correctly identified all 123 actual crisis cases.
- Transitional: Model missed 1 out of 13 actual crisis cases.
- Unstable or Unhoused: Model missed 1 out of 7 actual crisis cases.
- Unknown: Model correctly identified all 4 actual crisis cases.

The housing status analysis shows us that Stable Housed had the highest number of missed actual crisis cases by the model. But in comparison, the model manage to correctly identified all actual crisis cases for Doubled Up group. The difference between the highest recall of 100 percentage point and the lowest of 43.8 percentage point is a 56.2 percentage points. This have exceeds the 10 percentage point threshold and should be investigated further. All housing status also had fewer than 20 actual crisis cases in the validation set, so these results need to be interpreted with caution.

Region Mitigation - Excluding Region Feature

In [20]:
#Remove Region from Training and Validation Features
x_train_region = x_train.drop(columns = ["region"])
x_val_region = x_val.drop(columns = ["region"])

#Update Feature Lists
region_numeric_features = numeric_features.copy()
region_categorical_features = [
    feature for feature in categorical_features if feature != "region"
]

#Create a new Preprocessor
region_preprocessor = ColumnTransformer(transformers = [
    ("numeric", numeric_transformer, region_numeric_features),
    ("categorical", categorical_transformer, region_categorical_features)
])

#Create a new Random Forest Pipeline
region_mitigation_pipeline = Pipeline(steps=[
    ("preprocessor", region_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight="balanced",
        random_state=42
    ))
])

#Train Model
region_mitigation_pipeline.fit(x_train_region, y_train)

#Generate Validation Predictions Using the 0.30 Threshold
region_crisis_index = list(
    region_mitigation_pipeline.classes_
).index("crisis")
region_probabilities = region_mitigation_pipeline.predict_proba(
    x_val_region
)[:, region_crisis_index]
region_mitigation_predictions = (
    region_probabilities >= 0.30
).astype(int)

#Display Results
print("Region Mitigation Model Train Successfully.")
print("Validation Predictions:", len(region_mitigation_predictions))

Region Mitigation Model Train Successfully.
Validation Predictions: 360


Region Mitigation Results

In [22]:
#Evaluate the Region Mitigation Model
region_mitigation_audit = validation_audit[
    ["region", "actual_crisis"]
].copy()
region_mitigation_audit["predicted_crisis"] = (
    region_mitigation_predictions
)

region_mitigation_results = []

for group_name, group_data in region_mitigation_audit.groupby("region"):
    actual = group_data["actual_crisis"]
    predicted = group_data["predicted_crisis"]

    recall = recall_score(actual, predicted, zero_division=0)
    precision = precision_score(actual, predicted, zero_division=0)
    f1 = f1_score(actual, predicted, zero_division=0)

    region_mitigation_results.append({
        "Region": group_name,
        "Total Crisis Cases": int(actual.sum()),
        "Recall": recall,
        "False Negative Rate": 1 - recall,
        "Precision": precision,
        "F1 Score": f1
    })

region_mitigation_metrics = pd.DataFrame(
    region_mitigation_results
)

#Display Results
print("Region Mitigation Results:")
display(region_mitigation_metrics.round(3))

Region Mitigation Results:


,Region,Total Crisis Cases,Recall,False Negative Rate,Precision,F1 Score
0,Delta Basin,4,0.750,0.250,0.500,0.600
1,Gulf Coast,19,0.632,0.368,0.750,0.686
2,Inland Valley,15,0.867,0.133,0.448,0.591
3,Metro North,15,0.800,0.200,0.414,0.545


Region Mitigation Attempt: Removing Region Feature:

What Changed?
- Delta Basin: No change.
- Gulf Coast: Recall decreased by 5.2 percentage points.
- Inland Valley: Recall decreased by 6.6 percentage points.
- Metro North: No change.

For the first mitigation attempt, I removed the region feature from the Random Forest model to see whether it will reduce the recall differences between regions. The original recall gap was 24.9 percentage points and after removing the region feature the gap decreased slightly to 23.5 percentage points. The recall decreased in both Gulf Coast and Inland Valley, while Delta Basin and Metro North remained unchanged. Since the recall gap only improved by 1.4 percentage points, removing the region feature did not resolve the difference in the recall results. This mitigation attempt was not successful in addressing the region recall gap.

Housing Mitigation - Excluding Housing Status Feature

In [21]:
#Remove Housing Status from Training and Validation Features
x_train_housing = x_train.drop(columns=["housing_status"])
x_val_housing = x_val.drop(columns=["housing_status"])

#Update Feature Lists
housing_numeric_features = numeric_features.copy()
housing_categorical_features = [
    feature for feature in categorical_features
    if feature != "housing_status"
]

#Create a New Preprocessor
housing_preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, housing_numeric_features),
    ("categorical", categorical_transformer, housing_categorical_features)
])

#Create a new Random Forest Pipeline
housing_mitigation_pipeline = Pipeline(steps=[
    ("preprocessor", housing_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight="balanced",
        random_state=42
    ))
])

#Train Model
housing_mitigation_pipeline.fit(x_train_housing, y_train)

#Generate Validation Predictions Using the 0.30 Threshold
housing_crisis_index = list(
    housing_mitigation_pipeline.classes_
).index("crisis")
housing_probabilities = housing_mitigation_pipeline.predict_proba(
    x_val_housing
)[:, housing_crisis_index]
housing_mitigation_predictions = (
    housing_probabilities >= 0.30
).astype(int)

#Display Results
print("Housing Mitigation Model Train Successfully.")
print("Validation Predictions:", len(housing_mitigation_predictions))

Housing Mitigation Model Train Successfully.
Validation Predictions: 360


Housing Mitigation Results

In [23]:
#Evaluate the Housing Mitigation Model
housing_mitigation_audit = validation_audit[
    ["housing_status", "actual_crisis"]
].copy()
housing_mitigation_audit["predicted_crisis"] = (
    housing_mitigation_predictions
)
housing_mitigation_results = []

for group_name, group_data in housing_mitigation_audit.groupby(
    "housing_status"
):
    actual = group_data["actual_crisis"]
    predicted = group_data["predicted_crisis"]

    recall = recall_score(actual, predicted, zero_division=0)
    precision = precision_score(actual, predicted, zero_division=0)
    f1 = f1_score(actual, predicted, zero_division=0)

    housing_mitigation_results.append({
        "Housing Status": group_name,
        "Total Crisis Cases": int(actual.sum()),
        "Recall": recall,
        "False Negative Rate": 1 - recall,
        "Precision": precision,
        "F1 Score": f1
    })

housing_mitigation_metrics = pd.DataFrame(
    housing_mitigation_results
)

#Display Results
print("Housing Mitigation Results:")
display(housing_mitigation_metrics.round(3))

Housing Mitigation Results:


,Housing Status,Total Crisis Cases,Recall,False Negative Rate,Precision,F1 Score
0,Unknown,4,1.000,0.000,0.800,0.889
1,doubled_up,13,1.000,0.000,0.684,0.812
2,stable_housed,16,0.438,0.562,0.292,0.350
3,transitional,13,0.923,0.077,0.750,0.828
4,unstable_or_unhoused,7,0.429,0.571,0.600,0.500


Housing Mitigation Attempt: Removing Housing Status Feature:

What Changed?
- Unknown: No change.
- Doubled Up: No change.
- Stable Housed: No change.
- Transitional: No change.
- Unstable or Unhoused: Recall decreased by 42.8 percentage points.

For the second mitigation attempt, I removed the housing status feature from the Random Forest model to see if it would reduce the recall differences between the housing groups. The original recall gap was 56.2 percentage points. After removing the housing status feature, the gap increase slightly to 57.1 percentage points. Recall remained unchanged for Unknown, Doubled Up, Stable Housed, and Transitional. However, recall for Unstable or Unhoused decreased from 85.7% to 42.9%. Since the recall gap increased instead of decreasing, removing the housing status feature was not successful in addressing the housing status recall gap. This also negatively affected the model's ability to identify crisis cases in the Unstable or Unhoused group.

Threshold Adjustment Mitigation

In [24]:
#Test Different Thresholds
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
threshold_results = []

for threshold in thresholds:

    #Generate Predictions Using the Selected Threshold
    predictions = (val_probabilities >= threshold).astype(int)

    #Calculate Overall Performance
    overall_recall = recall_score(
        y_val_binary, predictions, zero_division=0
    )
    overall_precision = precision_score(
        y_val_binary, predictions, zero_division=0
    )
    overall_f1 = f1_score(
        y_val_binary, predictions, zero_division=0
    )

    #Create Temp Audit Data
    threshold_audit = validation_audit.copy()
    threshold_audit["predicted_crisis"] = predictions

    #Cal Recall Gaps by Region
    region_recalls = threshold_audit.groupby("region").apply(
        lambda group: recall_score(
            group["actual_crisis"],
            group["predicted_crisis"],
            zero_division=0
        )
    )
    region_gap = region_recalls.max() - region_recalls.min()

    #Cal Recall Gaps by Housing Status
    housing_recalls = threshold_audit.groupby(
        "housing_status"
    ).apply(
        lambda group: recall_score(
            group["actual_crisis"],
            group["predicted_crisis"],
            zero_division=0
        )
    )
    housing_gap = housing_recalls.max() - housing_recalls.min()

    threshold_results.append({
        "Threshold": threshold,
        "Overall Recall": overall_recall,
        "Overall Precision": overall_precision,
        "Overall F1": overall_f1,
        "Region Recall Gap": region_gap,
        "Housing Recall Gap": housing_gap
    })
threshold_comparison = pd.DataFrame(threshold_results)

#Display Results
print("Threshold Adjustment Comparison:")
display(threshold_comparison.round(3))

Threshold Adjustment Comparison:


,Threshold,Overall Recall,Overall Precision,Overall F1,Region Recall Gap,Housing Recall Gap
0,0.20,0.792,0.350,0.486,0.249,0.562
1,0.25,0.792,0.467,0.587,0.249,0.562
2,0.30,0.792,0.532,0.636,0.249,0.562
3,0.35,0.736,0.600,0.661,0.235,0.625
4,0.40,0.736,0.650,0.690,0.235,0.625
5,0.45,0.736,0.709,0.722,0.235,0.625
6,0.50,0.736,0.736,0.736,0.235,0.625


Threshold Adjustment Mitigation Attempt:

What We Learned:
- Thresholds 0.20–0.30 did not change either subgroup recall gap.
- Thresholds 0.35–0.50 slightly reduced the region gap from 24.9 to 23.5 percentage points.
    - The housing gap increased from 56.2 to 62.5 percentage points.
    - Overall recall also decreased from 79.2% to 73.6%.

For the third mitigation attempt, I have tested region and housing status at different classification thresholds from 0.20 to 0.50 to see whether adjusting the threshold would reduce the recall differences between the subgroups.

At a thresholds between 0.20 and 0.30, region and housing status recall gaps remained unchanged. At the thresholds between 0.35 and 0.50, the region recall gap decreased slightly from 24.9 to 23.5 percentage points, but the housing status recall gap increased from 56.2 to 62.5 percentage points. Overall recall also decrease from 79.2 to 73.6 at the higher thresholds.

Threshold adjustment also did not successfully address both of the flagged fairness gaps. The original threshold of 0.30 remains the better choice among the tested thresholds since it maintains the higher recall without increasing the housing status recall gap.

Fairness Audit Conclusion

The fairness audit found out there are differences in recall performance with the region, age group and housing status groups.

The age group analysis showed us that there is a large recall gap of 92.9 percentage points, but this difference was largely driven by the small number of actual crisis cases within the Adult and Young Adult groups. Since these groups had only 1 and 2 actual crisis cases, so it was not enough evidence to determine whether the model performs unfairly toward these groups. The region analysis show us there was a 24.9 percentage point recall gap between Inland Valley and Gulf Coast. The housing status analysis shows us a 56.2 percentage point gap between the highest and lowest recall groups. Both gaps have exceeded the 10 percentage point threshold and need to be look into further. But we need to note that all region and housing status subgroups had fewer than 20 actual crisis cases in the validation set, so these results should be interpreted with caution. The substance use flag analysis showed a 9 percentage point recall gap which was below the 10 percentage point threshold.

There was three mitigation strategies that were tested:
    - Removing region features
    - Removing housing status feature
    - Adjusting the classification threshold
None of these have successfully addressed the flagged recall gaps. Leaving the original Random Forest model at a threshold of 0.30 as the better choice among all the tested options since it maintained a higher recall without increasing the housing status recall gap.

These findings show us that the model still has fairness risk and that it needs to be looked into further. The model should not be used as the sole basis for deciding about the client outreach or care and need a professional judgment with human oversight to continue monitoring the subgroup performance.